In [ ]:
import os
from pathlib import Path

import pandas as pd
from src import analysis
from src.analysis import evaluation

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
# Models / architectures to compare
MODELS = [
    "S-FNO_m12x12_h24_l4_20260109_201346",
    "S-PI-FNO-divu-PS_m12x12_h24_l4_lamPhys1e-03_lamP5e-03_20260117_212344",
    "S-PI-FNO-divepsu-PS_m12x12_h24_l4_lamPhys1e-03_lamP5e-03_20260109_225428",
    # #
    # "M-FNO_m48x48_h64_l4_20260110_085608",
    # "M-UNO_m64x64_h32_l7_s1-05-05-1-1-2-2_mr0p495_20260116_022222",
    # "M-PI-FNO-divepsu-PS_m48x48_h64_l4_lamPhys1e-04_lamP5e-04_20260110_225103",
    # "M-PI-FNO-divepsu-SP_m48x48_h64_l4_lamPhys1e-04_lamP5e-04_20260119_011937",
    # #
    "FNO_m128x160_h64_l3_20260118_104314",
    "UNO_m64x64_h32_l7_s1-05-05-1-1-2-2_mr0p495_20260116_022222",
    "PI-FNO-divu-SP_m48x48_h64_l3_lamPhys1e-04_lamP7e-04_20260118_222756",
    "PI-UNO-divepsu-PS_m64x64_h32_l7_s1-05-05-1-1-2-2_mr0p495_lamPhys1e-04_lamP5e-04_20260117_175452",
]

MODEL_LABELS = {
    "S-FNO_m12x12_h24_l4_20260109_201346": "S-FNO",
    "S-PI-FNO-divu-PS_m12x12_h24_l4_lamPhys1e-03_lamP5e-03_20260117_212344": "S-PI-FNO-divu-PS",
    "S-PI-FNO-divepsu-PS_m12x12_h24_l4_lamPhys1e-03_lamP5e-03_20260109_225428": "S-PI-FNO-divεsu-PS",
    #
    # "M-FNO_m48x48_h64_l4_20260110_085608": "A-FNO",
    # "M-UNO_m64x64_h32_l7_s1-05-05-1-1-2-2_mr0p495_20260116_022222": "A-UNO",
    # "M-PI-FNO-divepsu-PS_m48x48_h64_l4_lamPhys1e-04_lamP5e-04_20260110_225103": "A-PI-FNO-divεsu-PS",
    # "M-PI-FNO-divepsu-SP_m48x48_h64_l4_lamPhys1e-04_lamP5e-04_20260119_011937": "A-PI-FNO-divεsu-SP",
    # #
    "FNO_m128x160_h64_l3_20260118_104314": "FNO",
    "UNO_m64x64_h32_l7_s1-05-05-1-1-2-2_mr0p495_20260116_022222": "UNO",
    "PI-FNO-divu-SP_m48x48_h64_l3_lamPhys1e-04_lamP7e-04_20260118_222756": "PI-FNO-divu-SP",
    "PI-UNO-divepsu-PS_m64x64_h32_l7_s1-05-05-1-1-2-2_mr0p495_lamPhys1e-04_lamP5e-04_20260117_175452": "PI-UNO-divεu-PS",
}

# Shared datasets (IDENTICAL for all models)
ID_DATASET = "lhs_var80_seed3001"
OOD_DATASET = "lhs_var120_seed4001"

show_id_evaluation = True
show_ood_evaluation = True

In [ ]:
# ======================================================================
# Paths
# ======================================================================
RAW_ROOT = Path("../../data/raw")
PROCESSED_ROOT = Path("../data/processed")


def run_or_load_artifacts_evaluation(
    *,
    model_name: str,
    dataset_name: str,
) -> pd.DataFrame:
    """Load or generate evaluation artifacts for one model on one dataset."""
    run_dir = PROCESSED_ROOT / model_name
    checkpoint_path = run_dir / "best_model_state_dict.pt"

    dataset_path = RAW_ROOT / dataset_name / "cases"

    save_root = run_dir / "analysis" / "id" if dataset_name == ID_DATASET else run_dir / "analysis" / "ood" / dataset_name

    parquet_path = save_root / f"{dataset_name}.parquet"

    if parquet_path.exists():
        print(f"[INFO] Loading {model_name} | {dataset_name}")
        return pd.read_parquet(parquet_path)

    print(f"[INFO] Creating artifacts: {model_name} | {dataset_name}")

    model, loader, processor, device = analysis.analysis_interference.load_inference_context(
        dataset_path=dataset_path,
        checkpoint_path=checkpoint_path,
        batch_size=1,
    )

    df, _ = analysis.analysis_artifacts.generate_artifacts(
        model=model,
        loader=loader,
        processor=processor,
        device=device,
        save_root=save_root,
        dataset_name=dataset_name,
    )

    return df


def make_model_label(model_name: str) -> str:
    """Map full model name to human-readable label."""
    if model_name in MODEL_LABELS:
        return MODEL_LABELS[model_name]
    return model_name

In [ ]:
# ======================================================================
# Load evaluation data
# ======================================================================
datasets_eval_id = {}
datasets_eval_ood = {}

for model_name in MODELS:
    print(f"\n=== {model_name} ===")

    if show_id_evaluation:
        # -----------------
        # ID
        # -----------------
        df_raw_id = run_or_load_artifacts_evaluation(
            model_name=model_name,
            dataset_name=ID_DATASET,
        )
        df_id = evaluation.evaluation_dataframe.build_eval_df(df_raw_id)
        label = make_model_label(model_name)
        datasets_eval_id[label] = df_id

    if show_ood_evaluation:
        # -----------------
        # OOD
        # -----------------
        df_raw_ood = run_or_load_artifacts_evaluation(
            model_name=model_name,
            dataset_name=OOD_DATASET,
        )
        df_ood = evaluation.evaluation_dataframe.build_eval_df(df_raw_ood)
        label = make_model_label(model_name)
        datasets_eval_ood[label] = df_ood

In [ ]:
if show_id_evaluation:
    panel_id = evaluation.evaluation_panel.build_evaluation_panel(
        datasets_eval=datasets_eval_id,
        title="Model Comparison ID",
        sections="all",
    )
    display(panel_id)

In [ ]:
if show_ood_evaluation:
    panel_ood = evaluation.evaluation_panel.build_evaluation_panel(
        datasets_eval=datasets_eval_ood,
        title="Model Comparison OOD",
        sections="all",
    )
    display(panel_ood)